# 08_03 Instruction fine-tuning (small)

*📖 Book: chapter 7 (fine-tuning to follow instructions)*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **after the main course**: teaching the adult to take requests
>
> Project files this connects to:
>
> - `data/homer/odyssey/english/`
> - `data/homer/iliad/english/`
> - `src/homer_ai/pretrained.py`
>
> **This notebook:** Ask GPT-2 a question and it just keeps writing. How do chat models learn to answer a request, and how far does a small model get on a laptop?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- Why a plain GPT doesn't answer requests, and what instruction fine-tuning changes
- What an instruction format is, and how to build a small instruction set from Butler's Homer
- How to batch texts of different lengths so that padding doesn't count in the loss
- How to score written answers, and what GPT-2 small can and can't learn in a minute

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **instruction fine-tuning** | fine-tuning on examples of a request followed by a good answer, so the model learns to answer and not just to continue | `What is the first word…` → `The first word is "Tell".` |
| **instruction format** | one fixed layout for every example: where the request goes, where the input goes, where the answer starts | `### Instruction:` … `### Response:` |
| **ignore index** | a special target value, -100, that tells the loss to skip that position | the targets of padding tokens |
| **exact match** | a strict score: an answer counts only if it's letter for letter the expected one | 27 of 30 |

## 3. The idea

### ❓ GPT-2 writes English. Why can't I just ask it something?

Because it was never taught that a question wants an answer. It learned one thing: what text usually comes next on a web page. Give it a question and it continues the page. Maybe with another question, maybe with something that only looks like an answer.

**Instruction fine-tuning** fixes that with examples. Each one is a request and a good answer, always in the same layout. After enough of them, "what usually comes next" after a request *is* an answer.

In the picture of [08_00](08_00_chapter_overview.ipynb): the adult speaks English, but nobody ever told them that at this desk people come with requests. We teach them to take requests politely. That's the same idea that turned plain language models into chat assistants, done here at toy size.

### ❓ What will our instructions be?

Three small jobs on single sentences from Butler's English Homer in `data/homer/`:

```
first   What is the first word of this sentence?                      → The first word is "Telemachus".
last    What is the last word of this sentence?                       → The last word is "did".
poem    Which poem is this sentence from, the Iliad or the Odyssey?   → The Odyssey.
```

I chose them for one reason: a program can work out every right answer from our own files. The first and last word are in the sentence, and the poem is the folder the sentence came from. So nothing in the data is made up.

They also differ in difficulty, which is what makes this interesting. For `first` the answer sits right at the start of the input. For `last` it has to be found at the end. And `poem` needs knowledge: who appears in which poem.

👉 **Predict first:** which of the three will GPT-2 small learn in one minute of training? All, some, none?

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: where do the sentences come from?

From the 48 English book files. We cut the text into sentences at `.`, `!` and `?`, and keep the ones that are 6 to 16 words long and have no quotation marks (so each is short and stands alone). Then we take 390 from each poem, in a random order fixed by a seed.

In [ ]:
import random
import re

from homer_ai.lessons import HOMER, load_book_files


def sentences(poem):
    found = []
    for book in load_book_files(HOMER / poem / "english"):
        for sentence in re.split(r"(?<=[.!?])\s+", book.replace("\n", " ")):
            n_words = len(sentence.split())
            plain = not set(sentence) & set('"“”')  # no quotation marks
            if 6 <= n_words <= 16 and sentence[0].isupper() and plain:
                found.append(sentence.strip())
    return found


rng = random.Random(123)
pairs = []  # (poem, sentence)
for poem in ("odyssey", "iliad"):
    found = sentences(poem)
    print(f"{poem:8} {len(found)} usable sentences, e.g. {found[0]!r}")
    rng.shuffle(found)
    pairs += [(poem, sentence) for sentence in found[:390]]
rng.shuffle(pairs)
print(len(pairs), "sentences in all, half from each poem")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

397 usable sentences in the Odyssey and 586 in the Iliad. We keep 390 of each, so 780, and the `poem` job is balanced: guessing gives 50%.

</details>

### ❓ Step 2: how does a sentence become an instruction example?

Each sentence gets one of the three jobs in turn, and a function writes the right answer. An example is a small dict: the instruction, the input (the sentence), and the output (the answer).

In [ ]:
TASKS = {
    "first": "What is the first word of this sentence?",
    "last": "What is the last word of this sentence?",
    "poem": "Which poem is this sentence from, the Iliad or the Odyssey?",
}


def right_answer(task, sentence, poem):
    found = re.findall(r"[A-Za-z'’-]+", sentence)  # the words, without punctuation
    if task == "first":
        return f'The first word is "{found[0]}".'
    if task == "last":
        return f'The last word is "{found[-1]}".'
    return f"The {poem.capitalize()}."


examples = []
for i, (poem, sentence) in enumerate(pairs):
    task = list(TASKS)[i % 3]
    examples.append(
        {
            "task": task,
            "instruction": TASKS[task],
            "input": sentence,
            "output": right_answer(task, sentence, poem),
        }
    )

test_examples, train_examples = examples[:90], examples[90:]
print(len(train_examples), "to train on,", len(test_examples), "kept for the test")
for example in test_examples[:3]:
    print()
    print("input :", example["input"])
    print("job   :", example["instruction"])
    print("answer:", example["output"])

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

690 examples to train on and 90 for the test, 30 of each job. The three printed ones show one job each, with the answer the program worked out.

</details>

### ❓ Step 3: what's the instruction format?

The model reads plain text, so the three parts of an example must become one string, laid out the same way every time. We use the layout Raschka's book uses (it comes from Stanford's Alpaca project):

```
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
What is the first word of this sentence?

### Input:
Telemachus saw her long before any one else did.

### Response:
The first word is "Telemachus".<|endoftext|>
```

Everything down to `### Response:` is the **prompt**. For training we add the answer and `<|endoftext|>`. That last token teaches the model to stop: after an answer, the next thing is the end.

When we ask a question later, we give only the prompt and let the model write on from `### Response:`.

In [ ]:
def prompt_of(example):
    return (
        "Below is an instruction that describes a task. "
        "Write a response that appropriately completes the request."
        f"\n\n### Instruction:\n{example['instruction']}"
        f"\n\n### Input:\n{example['input']}"
        "\n\n### Response:\n"
    )


print(prompt_of(train_examples[0]) + train_examples[0]["output"])

### ❓ Step 4: what does GPT-2 answer before any fine-tuning?

Let's ask it the three test questions above. `reply` gives the model the prompt, lets it write up to 14 new tokens with greedy writing, and returns only the new part.

In [ ]:
import tiktoken
import torch

from homer_ai.pretrained import load_gpt2

encoding = tiktoken.get_encoding("gpt2")
model = load_gpt2()  # GPT-2 small inside our own GPTModel (08_01); read from the cache

In [ ]:
from homer_ai.pretrained import continue_text

EOT = 50256  # <|endoftext|>


def reply(model, example):
    prompt = prompt_of(example)
    text = continue_text(model, encoding, prompt, 14, stop_id=EOT)
    return text[len(prompt) :].strip()


for example in test_examples[:3]:
    print("expected:", example["output"])
    print("GPT-2:   ", repr(reply(model, example)))
    print()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

It doesn't answer. For the first one it talks *about* the question (`The first word of this sentence is the first word of the response`), and for the other two it just repeats pieces of the input. It read the request as the start of a page and continued the page.

</details>

### ❓ Step 5: how do examples of different lengths go into one batch?

Padding again, as in [08_02](08_02_spam_classifier.ipynb), but now the job is next-token guessing, so the targets are the inputs moved one token on ([02_04](../02_text_to_numbers/02_04_sliding_window.ipynb)).

There's one catch. We don't want the model to be graded on guessing filler tokens. PyTorch's `cross_entropy` skips every position whose target is **-100** (its ignore index). So the targets get -100 wherever there's nothing real to guess:

```
a short text, 5 tokens, padded to 8:
inputs :  t1   t2   t3   t4   t5   pad   pad   pad
targets:  t2   t3   t4   t5   EOT  -100  -100  -100
                              ↑ learn to stop    ↑ not graded
```

Read it column by column: after `t5` the right next token is `<|endoftext|>`, and after that nothing counts.

In [ ]:
encoded = [encoding.encode(prompt_of(e) + e["output"]) for e in train_examples]
sizes = [len(row) for row in encoded]
print("tokens per example: shortest", min(sizes), " longest", max(sizes))


def make_batch(rows):
    length = max(len(row) for row in rows) + 1  # + 1 for <|endoftext|>
    inputs = torch.full((len(rows), length), EOT)
    targets = torch.full((len(rows), length), -100)
    for r, row in enumerate(rows):
        row = row + [EOT]
        inputs[r, : len(row)] = torch.tensor(row)
        targets[r, : len(row) - 1] = torch.tensor(row[1:])
    return inputs, targets


inputs, targets = make_batch([[11, 12, 13, 14, 15], [21, 22, 23]])  # two toy texts
print("inputs :", inputs.tolist())
print("targets:", targets.tolist())

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Two toy texts of 5 and 3 tokens:

```
inputs : [11, 12, 13, 14, 15, 50256]   [21, 22, 23, 50256, 50256, 50256]
targets: [12, 13, 14, 15, 50256, -100] [22, 23, 50256, -100, -100, -100]
```

Each targets row is its inputs row moved one on, it ends with 50256 (stop), and the rest is -100. The real examples are about 70 tokens long, mostly the fixed layout around a short sentence.

</details>

### ❓ Step 6: how do we train?

With the loop of [05_07](../05_training_on_the_odyssey/05_07_train_the_gpt.ipynb), unchanged: a batch of 8 random examples, the scores flattened, cross-entropy against the targets.

One thing is different from 08_02: **nothing is frozen**. Writing an answer uses the whole model, bottom to top, so we let all 163 million numbers move. That makes each step slow, about 0.7 seconds on my laptop. We do 80 steps, so the model sees 640 examples, a little under one pass over the training set. It takes about a minute.

👉 **Predict first:** the loss is the average over every token of the example, and most of those tokens are the fixed layout. Will the loss fall fast or slowly?

In [ ]:
import time

import torch.nn.functional as F

optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=0.1)
picker = torch.Generator().manual_seed(123)

model.train()
started = time.time()
for step in range(1, 81):
    chosen = torch.randint(0, len(encoded), (8,), generator=picker).tolist()
    inputs, targets = make_batch([encoded[i] for i in chosen])
    loss = F.cross_entropy(model(inputs).flatten(0, 1), targets.flatten())
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if step % 10 == 0:
        print(f"step {step:2} | loss {loss.item():.3f} | {time.time() - started:4.0f}s")
model.eval()
print(f"training took {time.time() - started:.0f} seconds")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Fast at first: about 1.24 at step 10 and 0.86 by step 30, then slowly on to about 0.63 at step 80. The quick drop is the model learning the layout, which is the same in every example and easy to predict. The slow part is the sentences themselves, which are new text every time.

</details>

### ❓ Step 7: does it answer now?

The same three test questions as in step 4.

In [ ]:
for example in test_examples[:3]:
    print("expected:", example["output"])
    print("model:   ", repr(reply(model, example)))
    print()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Now it answers, in the right form, and it stops after the answer: `The first word is "As".`, `The last word is "Ulysses".`, `The Odyssey.` That part, the politeness, it learned in a minute.

But only the first of the three is *right*. The last word was `rose`, and the sentence is from the Iliad. Three examples can't tell us much, so let's count.

</details>

### ❓ Step 8: how often is it right?

We score all 90 test examples with **exact match**: an answer counts only if it's letter for letter the expected one. That's strict, and for answers this short it's fair.

In [ ]:
right = {task: 0 for task in TASKS}
for example in test_examples:
    right[example["task"]] += reply(model, example) == example["output"]

for task, n in right.items():
    print(f"{task:6} {n:>2} of 30 exactly right")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

On my laptop:

```
first  28 of 30
last   15 of 30
poem   15 of 30
```

This is the honest picture of a small model.

- **first** works. The answer is the first thing after `### Input:`, so the model only has to copy from a fixed place.
- **last** works half the time. It has to find where the sentence ends and copy the word before the full stop. It learned the form of the answer, and often quotes a word from near the end, not the last one.
- **poem** is at 15 of 30, and guessing gives 15. It learned nothing here. It says `The Iliad.` or `The Odyssey.` in perfect form, with no idea which. To get this right it would need to know that Ulysses and Telemachus belong to one poem and Achilles and Hector to the other, and 230 examples in one minute didn't teach it that.

So instruction fine-tuning taught the *behaviour* quickly: take a request, answer in the expected form, stop. It didn't add knowledge or skill beyond what's easy. That's the limit of the adult picture I mentioned in 08_00: a person who understood the request wouldn't answer the poem question with a confident coin flip.

</details>

### ❓ What does it do with a request it never saw?

Our training set has exactly three instructions. Let's ask a fourth.

In [ ]:
new_request = {
    "instruction": "How many words are in this sentence?",
    "input": "Tell me, O Muse, of that ingenious hero.",
}
print(repr(reply(model, new_request)))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`The hero.` Short, confident, in the style of the answers it learned, and not an answer to the question. It didn't learn to follow instructions in general. It learned three patterns, and here it falls somewhere between them. Models that handle new requests were trained on a great many different instructions (the set in Raschka's book has 1,100 examples of many kinds; real assistants use far more), and they're much bigger.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Why not use the instruction set from Raschka's book?

I looked. His repository (`rasbt/LLMs-from-scratch`) has `instruction-data.json` with 1,100 examples, and the repository's licence is Apache 2.0, with a line added that leaves out the book's own text and pictures. That probably makes the data file fine to reuse, but "probably" isn't what I want for a course others will copy. So this notebook builds its own set from the Butler translations we already ship, where every answer is worked out by a few lines of code you can read above.

### ❓ Would more training fix the weak results?

Some of them. More steps would likely push `last` up, since the answer is in the input. `poem` needs the model to learn facts about names from 230 examples, which is slow going. You can try: run the training cell again for 80 more steps (the optimizer keeps its state) and score again. The book uses a model three times bigger and a more varied instruction set; this notebook stays small so that it runs in a couple of minutes.

### ❓ Why did we unfreeze everything here, but not in 08_02?

A classifier only needs the last block to gather one answer. Writing an answer token by token uses every layer, and the new behaviour (answer, then stop) touches all of them. The price is speed: 0.7 seconds a step, against 0.17 in 08_02.

### ❓ Is exact match too strict?

For free-form answers it would be. `It is the Odyssey.` is right but not an exact match. Bigger projects score answers with a second model, or by hand. Our answers have one fixed form, and the model learned that form, so exact match is fair here.

## 5. Recap

- A plain GPT continues text. Instruction fine-tuning shows it requests followed by answers, in one fixed format, until answering is what comes next.
- We built 780 examples from Butler's Homer with answers a program can work out, so nothing is invented.
- Padding targets are set to -100 so the loss skips them; `<|endoftext|>` after each answer teaches the model to stop.
- In one minute, GPT-2 small learned the form of the answers and the easy job well (28 of 30). The job that needs knowledge stayed at chance (15 of 30).
- It learned three patterns, not instructions in general.

**Next:** [08_04: LoRA](08_04_lora.ipynb).